# Index · Unit 4 — Multi-agent systems

Same shape as Units 1–3. For every component in **Section 5 of `index.md`**, two cells:

- **From scratch** — the mechanism by hand, no framework. Runs on plain Python /
  scripted *fake agents* (no API key needed) so you see every moving part.
- **SOTA** — how it's actually done today (role-prompted models, structured
  handoffs). Needs `OPENAI_API_KEY`.

Components: **1** orchestration patterns (sequential / parallel / supervisor) ·
**2** roles & specialization (planner / researcher / writer / critic) ·
**3** handoffs & delegation · **4** agent-to-agent communication (blackboard).

**Intuition for the whole unit:** one agent = one model call with a role and some
tools. A **multi-agent system** is just several of those wired together — the
only new ideas are *who runs when* (orchestration), *who is good at what*
(roles), *how control passes* (handoffs), and *how they share information*
(communication). Nothing here is more than functions calling functions.

Running example used throughout: **"write a short blurb about the Eiffel
Tower"**, handled by a planner → researcher → writer → critic team. The
researcher reads real facts from `research_notes.txt`.

## Setup

In [ ]:
import os, re, json

MODEL = "gpt-4o-mini"

print("✅ key set — SOTA cells will run" if os.environ.get("OPENAI_API_KEY")
      else "⚠️  no OPENAI_API_KEY — from-scratch cells still run; SOTA cells need a key")

def chat(messages, model=MODEL, **kwargs):
    from openai import OpenAI
    return OpenAI().chat.completions.create(
        model=model, messages=messages, **kwargs).choices[0].message

# an "agent" with a role is just a model call wrapped with a system prompt.
def agent(role_prompt, user_msg, **kwargs):
    return chat([{"role": "system", "content": role_prompt},
                 {"role": "user", "content": user_msg}], **kwargs).content

# shared knowledge the researcher can read (created alongside this notebook).
NOTES = open("research_notes.txt").read()
print("loaded notes:", len(NOTES), "chars")

## 1. Orchestration patterns

Orchestration = *who runs when*. Three canonical shapes:

- **sequential** — a pipeline; each agent's output feeds the next.
- **parallel** — fan out to independent agents, then combine.
- **supervisor / worker** — a router agent picks which worker handles the input.

**From scratch:** all three as plain function composition over fake agents.

In [ ]:
# From scratch — sequential, parallel, supervisor over trivial fake agents.
def upper(x):   return x.upper()
def reverse(x): return x[::-1]
def excite(x):  return x + "!!!"

def sequential(x, steps):
    for s in steps: x = s(x)
    return x

def parallel(x, workers):
    return [w(x) for w in workers]          # independent; combine however you like

def supervisor(x):
    route = "excite" if "?" in x else "upper"   # router picks a worker
    worker = {"excite": excite, "upper": upper}[route]
    print(f"  supervisor routed to: {route}")
    return worker(x)

print("sequential:", sequential("hello", [upper, reverse, excite]))
print("parallel  :", parallel("hello", [upper, reverse, excite]))
print("supervisor:", supervisor("are you there?"))

In [ ]:
# SOTA — a supervisor model routes to specialist agents.
WORKERS = {
    "math":    "You are a calculator. Return only the numeric answer.",
    "trivia":  "You are a trivia expert. Answer in one short sentence.",
    "writer":  "You are a copywriter. Reply with a catchy one-liner.",
}

def sota_supervisor(task):
    route = agent("Classify the task into exactly one of: math, trivia, writer. "
                  "Reply with only that word.", task).strip().lower()
    route = route if route in WORKERS else "trivia"
    print(f"  supervisor routed to: {route}")
    return agent(WORKERS[route], task)

for t in ["What is 12 * 8?", "Who painted the Mona Lisa?", "Slogan for a coffee shop"]:
    print(t, "->", sota_supervisor(t))

## 2. Roles & specialization

A **role** is just a different system prompt (and maybe different tools) giving
each agent a narrow job: **planner** (break down the goal), **researcher** (gather
facts), **writer** (produce output), **critic** (check quality). Specialization
beats one do-everything prompt because each agent stays focused.

**From scratch:** four fake specialists, each a small deterministic function.

In [ ]:
# From scratch — four specialists with distinct behaviors.
def planner(goal):      return [f"research: {goal}", f"write: {goal}", "review draft"]
def researcher(goal):   # find the matching TOPIC block, return its bullets
    subject = goal.split("about")[-1]
    keys = {w for w in re.findall(r"[a-z]+", subject.lower()) if w not in {"the","a","about"}}
    for block in NOTES.split("TOPIC:")[1:]:
        header = block.splitlines()[0].lower()
        if keys & set(re.findall(r"[a-z]+", header)):            # topic header matches subject
            return [l.strip("- ") for l in block.splitlines() if l.startswith("-")]
    return []
def writer(facts):      return "Eiffel Tower: " + "; ".join(facts[:2]) + "."
def critic(draft):      return ("OK" if len(draft) < 120 else "too long") , draft

goal = "write about the Eiffel Tower"
plan  = planner(goal);           print("plan   :", plan)
facts = researcher(goal);        print("facts  :", facts)
draft = writer(facts);           print("draft  :", draft)
print("critic :", critic(draft))

In [ ]:
# SOTA — the same four roles as system-prompt personas on a real model.
PLANNER   = "You are a planner. Break the goal into 3 terse numbered steps."
RESEARCHER= "You are a researcher. Using ONLY the provided notes, list 3 key facts as bullets."
WRITER    = "You are a writer. Write a single engaging sentence from the facts."
CRITIC    = "You are a critic. Reply 'APPROVED' or one concrete fix. Be strict about length."

def sota_team(goal):
    plan  = agent(PLANNER, goal)
    facts = agent(RESEARCHER, f"Notes:\n{NOTES}\n\nGoal: {goal}")
    draft = agent(WRITER, facts)
    review= agent(CRITIC, draft)
    return {"plan": plan, "facts": facts, "draft": draft, "review": review}

print(json.dumps(sota_team("write a blurb about the Eiffel Tower"), indent=2))

## 3. Handoffs & delegation

Instead of a fixed pipeline, an agent can **decide** to pass control to another
agent — a **handoff** — sending along a payload. A small **runner** loop dispatches
to whichever agent is named next, until someone says `done`. This is how dynamic
teams (and the OpenAI Agents SDK's handoffs) work.

**From scratch:** each agent returns `(next_agent, payload)`; a runner dispatches
until `done`.

In [ ]:
# From scratch — agents return handoffs; a runner dispatches them.
def triage(payload):
    return ("refund", payload) if "refund" in payload.lower() else ("faq", payload)
def refund(payload):   return ("done", f"Refund started for: {payload}")
def faq(payload):      return ("done", f"Answered FAQ for: {payload}")

AGENTS = {"triage": triage, "refund": refund, "faq": faq}

def run(start, payload, max_hops=5):
    current = start
    for _ in range(max_hops):
        nxt, payload = AGENTS[current](payload)
        print(f"  {current} --handoff--> {nxt}")
        if nxt == "done": return payload
        current = nxt
    return "(stopped: max hops)"

print(run("triage", "I want a refund for my order"))
print(run("triage", "What are your opening hours?"))

In [ ]:
# SOTA — the model chooses the next agent via a structured handoff.
AGENT_MENU = ["refund", "faq", "escalate", "done"]

def sota_handoff(payload, start="triage", max_hops=4):
    current, history = start, []
    for _ in range(max_hops):
        m = agent(
            f"You are the '{current}' agent in a support team. "
            f"Decide the next agent from {AGENT_MENU} and a short message. "
            'Reply JSON: {"next": "...", "message": "..."}',
            payload, response_format={"type": "json_object"})
        step = json.loads(m)
        history.append((current, step["next"]))
        print(f"  {current} --handoff--> {step['next']}")
        if step["next"] == "done": return {"result": step["message"], "path": history}
        current, payload = step["next"], step["message"]
    return {"result": "(max hops)", "path": history}

print(sota_handoff("My package arrived broken, I want my money back."))

## 4. Agent-to-agent communication — the blackboard

Agents coordinate by **sharing information**. The simplest, most common pattern is
a **blackboard**: a shared store every agent can read from and write to. Each
agent watches for work it can do, posts its result, and others build on it — no
direct point-to-point wiring needed (A2A protocols formalize this for agents
across processes/vendors).

**From scratch:** a dict blackboard; agents read inputs and post outputs to it.

In [ ]:
# From scratch — a shared blackboard the team reads/writes.
blackboard = {"goal": "write about the Eiffel Tower"}

def researcher_bb(bb):
    if "facts" in bb: return
    bb["facts"] = researcher(bb["goal"])            # reuse role fn from section 2
    print("  researcher posted: facts")
def writer_bb(bb):
    if "draft" in bb or "facts" not in bb: return
    bb["draft"] = writer(bb["facts"])
    print("  writer posted: draft")
def critic_bb(bb):
    if "review" in bb or "draft" not in bb: return
    bb["review"] = critic(bb["draft"])[0]
    print("  critic posted: review")

# run agents in rounds until the board stops changing (they self-coordinate).
team = [researcher_bb, writer_bb, critic_bb]
for _ in range(5):
    before = dict(blackboard)
    for a in team: a(blackboard)
    if blackboard == before: break
print("\nfinal blackboard:", json.dumps(blackboard, indent=2))

In [ ]:
# SOTA — agents post structured messages to a shared log and read each other's.
board = {"goal": "write a blurb about the Eiffel Tower", "messages": []}

def post(sender, content): board["messages"].append({"from": sender, "content": content})
def read_board(): return "\n".join(f"{m['from']}: {m['content']}" for m in board["messages"])

post("researcher", agent(RESEARCHER, f"Notes:\n{NOTES}\n\nGoal: {board['goal']}"))
post("writer",     agent(WRITER, f"Shared board:\n{read_board()}\n\nWrite the blurb."))
post("critic",     agent(CRITIC, f"Shared board:\n{read_board()}\n\nReview the latest draft."))

for m in board["messages"]: print(f"[{m['from']}] {m['content']}\n")

## Recap

A multi-agent system is just functions calling functions — the four ideas:

| component | mechanism (from scratch) | SOTA packaging |
|---|---|---|
| orchestration | sequential / parallel / supervisor via function composition | a supervisor model routing to specialist agents |
| roles | distinct small functions per specialist | distinct system-prompt personas on one model |
| handoffs | agents return `(next, payload)`; a runner loop | model emits a structured `{next, message}` handoff |
| communication | a shared dict blackboard, read/write in rounds | agents post/read structured messages on a shared board |

The through-line across all four units holds: each mechanism is a few lines of
plain Python (a loop, a dict, a router), and SOTA swaps the fake agent for a
role-prompted model with structured outputs. Frameworks (OpenAI Agents SDK,
CrewAI, AutoGen, LangGraph) package exactly these four patterns.

Next in `index.md`: **Unit 5 — Safety, guardrails & security** (Section 6).